# 全結合型ニューラルネットワークと数値解析

MLPの記述方法を復習します．    


**目標：AIが賢くなる原理を試す**

---

例題では，分類問題（MNISTデータセット）を解きます．  
演習では，AIを賢くして例題と同じ分類問題を解きます．

---
## この教材について

「3分で学ぶPyTorch」シリーズの **MLP（多層パーセプトロン） 基礎編（第2回）** の演習パートです。

このノートブックは**演習ファイル（task）**です。`【TASK】` と書かれた箇所を埋めて実行してください。詰まったときは同じ回の回答ファイル（ans）を参照してください。

この教材と関連記事は note で無料公開しています。
シリーズ一覧: https://note.com/technosend/m/m84d841b6d067

---

プログラミングパートにさきがけて，下記の3つの説明をします．

- 全体の流れ：牛肉の格付け・値段決めを振り返ってPyTorchを用いたAIプログラミングの流れを整理
- MNISTデータセット：今回取り扱うデータセットの簡単に解説
- GPUの使い方：処理を高速化するためのColabの設定などを解説

その後，改めて例題・演習に取り組みます．


---



## 全体の流れ

牛肉の格付け・値段決めを振り返るとこんな順番になっている．  
1. ライブラリのインポート  
2. ニューラルネットワークの定義
3. 誤差関数と最適化器のインスタンス宣言（誤差関数・最適化器の設定）
4. ニューラルネットワークへのデータ入力と教師データ（データセットの準備）
5. 誤差逆伝播とパラメータの更新（学習）

AIの原理とプログラミングの順番を紐付けてあり，  
PyTorchに限らず他のライブラリやパッケージでもある程度同じ見方をすることができる．  

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="350" src="https://drive.google.com/thumbnail?id=11qWQuOgveLxhLS8N1_w5x4oEL858RsSb&sz=w400">
  



## MNISTデータセット

- 手書き数字データセット
- グレースケール（1チャネル）
- 縦横28×28ピクセル
- 0〜9までの10クラス
- 学習用データ：60,000（枚数はクラスによってばらつきがある）
- テスト用データ：10,000（枚数はクラスによってばらつきがある）
- 【解けてもすごいとは言えないが，解けなければなんでもない】というデータセット

[THE MNIST DATABASE of handwritten digits](http://yann.lecun.com/exdb/mnist/)

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="200" src="https://drive.google.com/thumbnail?id=1egOQqlalPdw6r-93wL5zte3--pVhU_9T&sz=w400">



## GPUを使うための手順

デバイスインスタンスを宣言して，torch.Tensor型（以降，Tensor型）の変数をGPUに渡す．

---



### GPUを使うための手順のコード

In [ ]:
import torch
# 1. デバイスインスタンスを宣言
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 2. Tensor型変数をGPUに渡す
a=torch.tensor([5, 4, 0])
print(a)
b = a.to(device)
print(b)

### 0. 前準備
- Colabの設定を変更


- Colabのランタイムを変更してGPUを使用可能なものにする
- 無料プランのアカウントの場合，同時に複数のColabファイルでGPUを使用することはできない
- 手順は下記の通り
    1. ランタイムをクリック
    2. ランタイムのタイプを変更をクリック
    3. ハードウェアアクセラレータのプルダウンを開ける
    4. GPUを選択
    5. 保存をクリックをクリック

<font color="blue">【TASK】</font>手順に従ってGPUを使用可能にしましょう

### 1. デバイスインスタンスを宣言

- deviceという名前でデバイスインスタンス用の変数を宣言

    ```python
    device = # 【TASK】デバイスインスタンスを宣言
    ```

- デバイスインスタンスは[torch.device](https://pytorch.org/docs/stable/tensor_attributes.html#torch.torch.device)クラスを用いて宣言
- GPUを使う場合は"cuda:0"，CPUを使う場合は"cpu"を引数として与える  
    ※ `torch.cuda.is_available()` でGPU有無を自動判定するのが推奨（2022年以降のColab対応）
- GPUが複数ある場合は，torch.device("cuda:1")，torch.device("cuda:2")のようにする

    ```python
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    # 第1引数：デバイスタイプ（"cpu" or "cuda"）
    ```

<font color="blue">【TASK】</font>デバイスインスタンスを宣言しましょう

### 2. Tensor型変数をGPUに渡す



- Tensor型変数から[to()](https://pytorch.org/docs/1.9.1/generated/torch.Tensor.to.html)を呼び出して引数与えたデバイスにデータを渡すことができる  
  引数に与えられるtorch.dtypeやtorch.deviceの種類は[こちら](https://pytorch.org/docs/stable/tensor_attributes.html)

    ```python
    a = torch.tensor([1])
    a.to(device)
    # 第1引数：dtypeやデバイスの指定(torch.dtype， torch.device)
    ```

<font color="blue">【TASK】</font>Tensor型変数を宣言してGPUに渡しましょう  
Tensor型変数はどんな値でも構いません

## 例題 MNISTデータセット

**全結合層1層のMLPの作成**  
28×28ピクセルのグレースケール画像を入力とし，そのラベル（10出力）を出力する分類問題を解く．

---



### 例題1. ライブラリのインポート
深層学習演算ライブラリPyTorchなどのライブラリ，パッケージ，モジュールをインポートする．  

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=14XdT7XWs6JzTil6JhZZw7aM3VpAxdYH2&sz=w400">



---

#### 例題1のコード

In [ ]:
# 例題1. ライブラリのインポート

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

#### 今回使うパッケージ一覧

  
  - torch：多次元テンソルのデータ構造とそのテンソルのための算術演算が組み込まれたパッケージ
  - torch.nn：ニューラルネットワークを定義するためのパッケージ
  - [torch.nn.functional](https://pytorch.org/docs/stable/nn.functional.html)：様々な関数が含まれているパッケージ  
    活性化関数やプーリング，正則化のための関数などが含まれている  
    Fという略称を与えていることが多い  
    ReLU関数は[relu()](https://pytorch.org/docs/stable/generated/torch.nn.functional.relu.html#torch.nn.functional.relu)，シグモイド関数は[sigmoid()](https://pytorch.org/docs/stable/generated/torch.nn.functional.sigmoid.html#torch.nn.functional.sigmoid)で呼び出せる（[relu()の詳細](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html#torch.nn.ReLU)，[sigmoid()の詳細](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html#torch.nn.Sigmoid)）
  - torch.optim：最適化器を宣言するためのパッケージ
  - [torchvision](https://pytorch.org/vision/stable/index.html)：画像処理のためのパッケージ  
    さまざまなデータセットや，ネットワークモデルが含まれている
  - [torchvision.transforms](https://pytorch.org/vision/stable/transforms.html)：データセットを整形するための関数が含まれているパッケージ  
  transformsと略すことが多い  

<font color="blue">【TASK】</font>パッケージをインポートしましょう

### 例題2. ニューラルネットワーククラスの定義

ニューラルネットワーククラスを定義して，そのクラスのインスタンスを宣言する．

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1jl1W_RW7HrU0ksk1a0XrSq6CyldXF4qZ&sz=w400">


---


#### 例題2のコード

In [ ]:
# 例題2. ニューラルネットワークの定義

# 1. ニューラルネットワーククラスの定義
class GrayImageClassifier(nn.Module):
    def __init__(self):
        super(GrayImageClassifier, self).__init__()
        # 【TASK】全結合層を定義

    def forward(self, x):
        # 【TASK】順伝播のパスを定義
        return x

# 2. インスタンスの宣言
gray_image_classifier = # 【TASK】ニューラルネットワーククラスのインスタンスの宣言
device = # 【TASK】GPUの指定
# 【TASK】GPUにセットアップ

- \_\_init__()では，層間の結合方法を定義  

    - super()を呼び出す  
    - 一つの層につき一つ，nnパッケージ内のクラスインスタンスを宣言  
    - 全結合層はnn.Linearクラスを用いて宣言  
    ```python
    self.fc1 = nn.Linear(784, 10)
    # 第1引数：入力(int)
    # 第2引数：出力(int)
    ```
    <font color="blue">【TASK】</font>MLPをnn.Linearクラスを用いて定義しましょう  
    ニューラルネットワークの構成は下記の通りです  
        - 全結合層1：入力784，出力10  

- forward()は外部からの入力データxを受け取り，順伝播を行う  
    - 全結合層の順伝播は，nn.Linearクラスのインスタンスの呼び出しで実行  
    ```python
    x = self.fc1(x)
    ```  
    <font color="blue">【TASK】</font>順伝播のパスを定義しましょう  
    28×28ピクセルのグーレースケール画像を入力すると想定したとき順伝播のパスの構成は下記の通りです  
        1. 全結合層1  



#### 1. ニューラルネットワーククラスの定義  

- nn.Moduleを継承したクラス「GrayImageClassifier」を定義  
- \_\_init\_\_()とforward()を定義
```python
class GrayImageClassifier(nn.Module):
    def __init__(self):
        super(GrayImageClassifier, self).__init__()
        # 【TASK】全結合層を定義
    def forward(self, x):
        # 【TASK】順伝播のパスを定義
        return x
```


#### 2. インスタンスの宣言

- gray_image_classifierという名前でインスタンスを宣言
    ```python
    gray_image_classifier = # 【TASK】ニューラルネットワーククラスのインスタンスの宣言
    ```

- GPUにセットアップ

    ```python
    device = # 【TASK】GPUの指定
    # 【TASK】GPUにセットアップ
    ```

<font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスを宣言しましょう  


- Moduleクラスにもto()がある
- Tensor型変数同様にto()を使ってGPUにデータを渡すことができる
- 学習時にGPUを使う場合は，ニューラルネットワーククラスのインスタンスをGPUに渡す

<font color="blue">【TASK】</font>ニューラルネットワーククラスのインスタンスをGPUにセットアップしましょう  
- `torch.cuda.is_available()` でGPU有無を判定し，利用可能な場合は"cuda:0"，利用不可の場合は"cpu"を指定しましょう
- to()を使ってGPUにセットアップしましょう


### 例題3. 誤差関数・最適化器の設定

誤差関数と最適化器を宣言する．

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1oS8_oitSvcQ9f5_uYMrDXyC3P_vrM7jp&sz=w400">


---

#### 例題3のコード

In [ ]:
# 例題3. 誤差関数・最適化器の設定

# 1. 誤差関数の宣言
criterion = # 【TASK】誤差関数の宣言

# 2. 最適化器の宣言
optimizer_gray_image_classifier = # 【TASK】最適化器の宣言

#### 1. 誤差関数の宣言

- クロスエントロピー誤差を計算するcriterionを宣言

    ```python
    criterion = # 【TASK】誤差関数の宣言
    ```


- クロスエントロピー誤差関数は[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)クラスを用いて宣言
- nn.CrossEntropyLoss()で宣言したインスタンスは，二つの引数のクロスエントロピー誤差を返す  
  ```python
  criterion = nn.CrossEntropyLoss()
  ```

<font color="blue">【TASK】</font>誤差関数を宣言しましょう  
クロスエントロピー誤差関数を使いましょう  

#### 2. 最適化器の宣言

- Adamを計算するoptimizer_gray_image_classifierを宣言

    ```python
    optimizer_gray_image_classifier = # 【TASK】最適化器の宣言
    ```

- Adamは[optim.Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#torch.optim.Adam)クラスを用いて宣言  
    ```python
    # 第1引数：ニューラルネットワークのパラメータ
    optimizer_gray_image_classifier = optim.Adam(gray_image_classifier.parameters())
    ```

<font color="blue">【TASK】</font>最適化器を宣言しましょう  
Adamを使いましょう  
引数の構成は下記の通りです  
- ニューラルネットワークのパラメータgray_image_classifierのパラメータ

### 例題4. データセットの準備  

MNISTデータセットを読み込み，データの整形やミニバッチの設定，データローダーの作成を行う．


<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=17fS-oMI83rSL7SxN_GyKHjxP-FO-R3aQ&sz=w400">


---



#### 例題4のコード

In [ ]:
# 例題4. データセットの準備

# 1. 整形方法を決定
transform = # 【TASK】データ整形の方法

# 2. データセットの読み込み
train_set = # 【TASK】学習データの読み込み
test_set = # 【TASK】テストデータの読み込み

# 3. ミニバッチの設定とデータローダーの作成
batch_size = # 【TASK】バッチサイズ
train_loader = # 【TASK】学習データのデータローダー
test_loader = # 【TASK】テストデータのデータローダー

#### 1. データ整形の設定

- データ整形の方法を指定するtransformを宣言
```python
transform  = # 【TASK】データ整形の方法
```

  - transformという変数名で，[transforms.Compose](https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.Compose)クラスのインスタンスを宣言
  - transforms.Composeクラスのインスタンスには整形の指示をlist形式で与える
    - データをPyTorchで扱えるようにする[transforms.ToTensor](https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.ToTensor)クラスと平均値と標準偏差値を指示できる[transforms.Normalize](https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.Normalize)クラスを与える
    - transforms.Normalizeクラスには引数として平均値，標準偏差値のタプルを与える  
      ```python
      transforms.Normalize((0.5,), (0.5,))
      # 第1引数：整形後のデータの平均値，データのチャネルごとにタプルで与える（tuple）
      #　　第2引数：整形後のデータの標準偏差値，データのチャネルごとにタプルで与える（tuple）
      ```

  ```python
  transform = transforms.Compose([
                                  transforms.ToTensor(),
                                  transforms.Normalize((0.5,), (0.5,))
                                  ])
  # 引数：1つ以上の整形の指示（transformsモジュールのクラス）
  ```

<font color="blue">【TASK】</font>データ整形の方法を指定しましょう  
引数の構成は下記の通りです  
- ComposeクラスにはToTensorクラスとNormalizeクラスを与える
- 平均値と標準偏差値はどちらも0.5


#### 2. データセットの読み込み  
- train_setという名前で学習データ用の変数を宣言
- test_setという名前でテストデータ用の変数を宣言
```python
train_set = # 【TASK】学習データの読み込み
test_set = # 【TASK】テストデータの読み込み
```

  - MNISTデータセットの読み込みは[torchvision.datasets.MNIST](https://pytorch.org/vision/stable/datasets.html#mnist)クラスを用いて行う  
  ```python
train_set = torchvision.datasets.MNIST(root="./", train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root="./", train=False, download=True, transform=transform)
  # 第1引数：読み込むデータのディレクトリの指定(str)
  # 第２引数：学習するかどうか(bool)
  # 第3引数：ダウンロードするかどうか(bool)
  # 第４引数：データ整形の設定(torchvision.transforms)
  ```

<font color="blue">【TASK】</font>データセットの読み込みを行いましょう  
読み込みの設定は下記の通りです  
- 学習データ
    - ディレクトリ："./"
    - 学習を行う
    - ダウンロードを行う
    - データ整形は上で宣言したtransformを使う
- テストデータ
    - ディレクトリ："./"
    - 学習は行わない
    - ダウンロードを行う
    - データ整形は上で宣言したtransformを使う

#### 3. ミニバッチの設定とデータローダーの作成  

- 任意のミニバッチの設定でデータを読み込むデーターローダーを作成
- batch_sizeという名前でバッチサイズ用の変数を宣言  
  ※バッチサイズはミニバッチ1つに含まれるデータの数
- train_loaderという名前で学習データのデータローダー用の変数を宣言
- test_loaderという名前でテストデータのデータローダー用の変数を宣言  
```python
batch_size = # 【TASK】バッチサイズ
train_loader = # 【TASK】学習データのデータローダー
test_loader = # 【TASK】テストデータのデータローダー
```

- ミニバッチの設定は[torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)クラスを用いて行う

    ```python
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)
    # 第1引数：読み込むデータ(Dataset)
    # 第2引数：バッチサイズ(int)
    # 第3引数：シャッフルするかどうか(bool)
    ```

<font color="blue">【TASK】</font>ミニバッチの設定をしましょう
- バッチサイズ20
- 学習データのデータローダー
    - 読み込むデータtrain_set
    - シャッフルする
- テストデータのデータローダー
    - 読み込むデータtest_set
    - シャッフルしない



### 例題5. 学習 

教師データの誤差を計算し，パラメータを更新する．

<!-- 旧形式リンク変換済み (Google Drive 2023年仕様変更対応) -->
<img width="400" src="https://drive.google.com/thumbnail?id=1A6TjumoBejWpKGvD6TDQeivN1tpeEBp4&sz=w400">


---


#### 例題5のコード

In [ ]:
# 1. 学習ループの作成
epochs = # 【TASK】エポック数

# このfor文の中の処理はエポック単位で実行される
# エポック＝学習データを一度使い回すこと
# エポックのループ
for epoch in # 【TASK】エポック:
    # 学習誤差を確認するための変数の初期化
    train_loss = 0.0
    # 全てのミニバッチの誤差計算の結果を合計して
    # 学習データ全体の誤差を計算としたい

    # このfor文の中の処理はミニバッチの単位で実行される
    # 学習データのデータローダーのループ
    for data in # 【TASK】学習データのデータローダー:

        # 2. ニューラルネットワークへのデータの入力
        # 画像とラベルに分割
        inputs, labels = data
        inputs = # 【TASK】GPUにセットアップ
        labels = # 【TASK】GPUにセットアップ
        # バッチ数×データサイズに整列し直す
        inputs = # 【TASK】データの整列
        outputs = # 【TASK】ニューラルネットワークからの出力

        # 3. 誤差逆伝播とパラメータの更新
        # 【TASK】パラメータの微分値を初期化
        loss = # 【TASK】誤差の計算
        # 【TASK】誤差逆伝播
        # 【TASK】パラメータの更新

        # 学習誤差の保存
        # ミニバッチごとの学習誤差を合計する
        train_loss += loss.item()

    # テスト（学習と同じなので説明は割愛）
    # テスト誤差を確認するための変数の初期化
    test_loss = 0.0
    # テストデータのデータローダーのループ
    for data in test_loader:
        inputs, labels = data
        inputs = inputs.view([batch_size, 784])
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = gray_image_classifier(inputs)
        test_loss += criterion(outputs, labels).item()

    print("epoch:{}".format(epoch+1))
    # [学習・テストデータ1つ分の誤差] = [ミニバッチごとの学習・テスト誤差の合計] / [ミニバッチへの分割数]
    print("学習誤差：{}".format(train_loss / len(train_loader)))
    print("テスト誤差：{}".format(test_loss / len(test_loader)))    

#### 1. 学習ループの作成  
- epochsという名前でエポック用の変数を宣言
- 外側がエポック，内側がミニバッチの学習ループを作成
```python
epochs = # 【TASK】エポック数
for epoch in # 【TASK】エポック
    for data in # 【TASK】学習用ミニバッチ
    for data in # 【TASK】テスト用ミニバッチ
```

<font color="blue">【TASK】</font>学習ループを作成しましょう  
ループの設定は下記の通りです
  - エポック
      - エポック数2
      - [range](https://docs.python.org/ja/3/library/stdtypes.html#range)クラスを使ってループさせましょう
        ```python
        range(start, stop, step)
        # 第1引数：初期値
        # 第2引数：最終値
        # 第3引数：繰り返し加算する値，負の値も取れる
        # ※一つだけ値を与えるとそれはstopに与えられ，start=0, step=1になる
        ```
  - 学習用ミニバッチ
    - 学習用ミニバッチtrain_loader
    - [for](https://docs.python.org/ja/3/reference/compound_stmts.html#for)を使ってループさせましょう
      ```python
      for i in list:
        print(i)
      # i：取り出される値を保持する変数， iはindexのiで，iではなく適当な名前でよい
      # list：文字列や配列など複数の値を持つもの
      ```
  - テスト用ミニバッチ
    - テスト用ミニバッチtest_loader
    - forを使ってループさせましょう
    

#### 2. ニューラルネットワークへのデータの入力  
- GPUにセットアップ
- データの整列
- outputsという名前で出力用の変数を宣言
```python
# 画像とラベルに分割
inputs, labels = data
inputs = # 【TASK】GPUにセットアップ
labels = # 【TASK】GPUにセットアップ
inputs = # 【TASK】データの整列
outputs = # 【TASK】ニューラルネットワークからの出力
```

  - 学習用ミニバッチのループで取得したデータを画像(inputs)とラベル(labels)に分割  
  - 画像データは28*28の2次元なので784の1次元に整列  
  - データの整列には[view()](https://pytorch.org/docs/stable/tensor_view.html#tensor-view-doc)を使う
  ```python
  t = torch.tensor([[0, 1], [2, 3]])
  b = t.view(4, 1)
  # 引数：整列後の形状
  # ※あくまで整列なので，元の形状と同じデータ量である必要がある
  ```
  - ニューラルネットワーククラスのインスタンスに引数を与えて出力を取得  
    
<font color="blue">【TASK】</font>ニューラルネットワークへデータを入力しましょう

#### 3. 誤差逆伝播とパラメータの更新  
- パラメータの微分値を初期化
- loss という名前で誤差計算の結果用の変数を宣言
- 誤差逆伝播
- パラメータの更新
```python
# 【TASK】パラメータの微分値を初期化
loss = # 【TASK】誤差の計算
# 【TASK】誤差逆伝播
# 【TASK】パラメータの更新
```



- パラメータの微分値の初期化は最適化器の持つzero_grad()
- 誤差計算はnn.MSELossクラスやnn.CrossEntropyLossクラスなどの誤差関数クラスのインスタンスに引数を二つ与えて行う
```python
loss = criterion(outputs, labels)
# 第1引数：ニューラルネットワークの出力
# 第2引数：教師データ
```

- 誤差逆伝播はbackward()
- パラメータの更新は最適化器の持つstep()

<font color="blue">　【TASK】</font>誤差逆伝播とパラメータの更新を行いましょう





## 演習 MNISTデータセット

**例題よりも層数の多いMLPの作成**

例題よりも層数の多いニューラルネットワークを用意したり，エポック数を増やしたりして分類問題を解く．

### 演習1. ニューラルネットワークの層数を増やす



### 演習1のコード

In [ ]:
## 層数を増やしたバージョン

# 演習1. ニューラルネットワークの層数を増やす

# 1. ニューラルネットワーククラスの定義
class GrayImageClassifier(nn.Module):
    def __init__(self):
        super(GrayImageClassifier, self).__init__()
        # 【TASK】全結合層を定義

    def forward(self,x):
        # 【TASK】順伝播のパスを定義 

# 2. インスタンスの宣言
# ニューラルネットワーククラスのインスタンスの宣言
gray_image_classifier = GrayImageClassifier()
# 【TASK】GPUの指定
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# 【TASK】GPUにセットアップ
gray_image_classifier.to(device)



- ReLU関数はnn.functional.reluクラスを用いて宣言
- 中間層にはReLU関数が必要

<font color="blue">【TASK】</font>ニューラルネットワークの層数を増やしてみましょう  
- ニューラルネットワークの構成
 - 全結合層1：入力784，出力100  
 - 全結合層2：入力100，出力10  
- 順伝播のパス
 1. 全結合層1
 2. ReLU関数
 3. 全結合層2

- ニューラルネットワークのパラメータが書き換えられた
- ニューラルネットワークのパラメータは最適化器にセットしておく必要がある

<font color="blue">【TASK】</font>学習させて差分を見てみましょう  
手順は下記の通りです  
1. 演習1のコードを実行
2. 例題3のコードを実行
3. 例題5のコードを実行
4. 誤差の違いを確認する


### 演習2. エポック数を増やす

<font color="blue">【TASK】</font>エポック数を増やしてみましょう
- 例題5のコードで設定しているエポック数を10に書き換える

- Colabでは実行記録が残っている
- 学習し直すのでニューラルネットワークのパラメータを初期化したい
- ニューラルネットワークのパラメータは最適化器にセットしておく必要がある

<font color="blue">【TASK】</font>学習させて差分を見てみましょう  
手順は下記の通りです  
1. 演習1のコードを実行
2. 例題3のコードを実行
3. 例題5のコードを実行
4. 誤差の違いを確認する

# まとめ

今回は，PyTorchを用いたAIプログラミングの流れを整理しつつ，例題でグレースケール画像を10クラスに分類しました．演習では，ニューラルネットワークの層を増やしたり，エポックを増やしたりして，精度が向上する様子を確かめました．  
次の段階として，さらに層を増やしてみたり，ニューロン数を変更してみたり，エポックだけでなくバッチサイズも大きくしたりして精度がどのように変わるのか試すことが挙げられます．  
余力があればこれらの取り組みも行ってみてください．